# M-Shots Learning

In this notebook, we'll explore small prompt engineering techniques and recommendations that will help us elicit responses from the models that are better suited to our needs.

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

# Formatting the answer with Few Shot Samples.

To obtain the model's response in a specific format, we have various options, but one of the most convenient is to use Few-Shot Samples. This involves presenting the model with pairs of user queries and example responses.

Large models like GPT-3.5 respond well to the examples provided, adapting their response to the specified format.

Depending on the number of examples given, this technique can be referred to as:
* Zero-Shot.
* One-Shot.
* Few-Shots.

With One Shot should be enough, and it is recommended to use a maximum of six shots. It's important to remember that this information is passed in each query and occupies space in the input prompt.



In [2]:
# Function to call the model.
def return_OAIResponse(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=1,
        )

    return (response.choices[0].message.content)

In this zero-shots prompt we obtain a correct response, but without formatting, as the model incorporates the information he wants.

In [3]:
#zero-shot
context_user = [
    {'role':'system', 'content':'You are an expert in F1.'}
]
print(return_OAIResponse("Who won the F1 2010?", context_user))

Sebastian Vettel won the F1 World Championship in 2010. He was driving for the Red Bull Racing team.


For a model as large and good as GPT 3.5, a single shot is enough to learn the output format we expect.


In [4]:
#one-shot
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2000 f1 championship?
     Driver: Michael Schumacher.
     Team: Ferrari."""}
]
print(return_OAIResponse("Who won the F1 2011?", context_user))

The 2011 F1 World Championship was won by Sebastian Vettel, driving for Red Bull Racing.


Smaller models, or more complicated formats, may require more than one shot. Here a sample with two shots.

In [5]:
#Few shots
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Driver: Fernando Alonso.
Team: Renault.


In [6]:
print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton.
Team: Mercedes.


We've been creating the prompt without using OpenAI's roles, and as we've seen, it worked correctly.

However, the proper way to do this is by using these roles to construct the prompt, making the model's learning process even more effective.

By not feeding it the entire prompt as if they were system commands, we enable the model to learn from a conversation, which is more realistic for it.

In [7]:
#Recomended solution
context_user = [
    {'role':'system', 'content':'You are and expert in f1.\n\n'},
    {'role':'user', 'content':'Who won the 2010 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Sebastian Vettel. \nTeam: Red Bull. \nPoints: 256. """},
    {'role':'user', 'content':'Who won the 2009 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Jenson Button. \nTeam: BrawnGP. \nPoints: 95. """},
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton. 
Team: Mercedes. 
Points: 413.


We could also address it by using a more conventional prompt, describing what we want and how we want the format.

However, it's essential to understand that in this case, the model is following instructions, whereas in the case of use shots, it is learning in real-time during inference.

In [8]:
context_user = [
    {'role':'system', 'content':"""You are and expert in f1.
    You are going to answer the question of the user giving the name of the rider,
    the name of the team and the points of the champion, following the format:
    Drive:
    Team:
    Points: """
    }
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton
Team: Mercedes
Points: 413


In [9]:
context_user = [
    {'role':'system', 'content':
     """You are classifying .

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Classification: 

   - Driver: Fernando Alonso.
   - Team: Renault.


Few Shots for classification.


In [10]:
context_user = [
    {'role':'system', 'content':
     """You are an expert in reviewing product opinions and classifying them as positive or negative.

     It fulfilled its function perfectly, I think the price is fair, I would buy it again.
     Sentiment: Positive

     It didn't work bad, but I wouldn't buy it again, maybe it's a bit expensive for what it does.
     Sentiment: Negative.

     I wouldn't know what to say, my son uses it, but he doesn't love it.
     Sentiment: Neutral
     """}
]
print(return_OAIResponse("I'm not going to return it, but I don't plan to buy it again.", context_user))

Sentiment: Neutral


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [11]:
context_user = [
    {'role':'system', 'content':
     """You are an expert in classifying movies by genre based on their plot description.

     Plot: A group of astronauts gets lost in space and must find their way back to Earth while dealing with limited oxygen.
     Genre: Science Fiction

     Plot: Two people meet in a coffee shop and slowly fall in love over the course of a year.
     Genre: Romance

     Plot: A detective investigates a series of mysterious murders in a small town.
     Genre: Mystery/Thriller
     """}
]
print(return_OAIResponse("Plot: A family moves into an old house and strange supernatural events begin to occur.", context_user))

Genre: Horror


In [12]:
context_user = [
    {'role':'system', 'content':
     """You are an expert in classifying recipes by difficulty level.

     Recipe: Mix flour and water, knead for 10 minutes, let rest for 1 hour, bake at 350°F.
     Difficulty: Easy

     Recipe: Prepare a beef wellington with mushroom duxelles, wrap in puff pastry, cook to perfect temperature.
     Difficulty: Advanced

     Recipe: Cook pasta, heat sauce, combine and add cheese.
     Difficulty: Beginner
     """}
]
print(return_OAIResponse("Recipe: Make a soufflé with beaten egg whites, careful temperature control, and precise timing.", context_user))

Difficulty: Advanced


In [13]:
context_user = [
    {'role':'system', 'content':
     """You are an expert in classifying email urgency levels.

     Email: Meeting scheduled for tomorrow at 10 AM to discuss quarterly results.
     Urgency: Medium

     Email: Server is down, affecting all customer transactions.
     Urgency: High

     Email: Weekly newsletter with company updates and announcements.
     Urgency: Low
     """}
]
print(return_OAIResponse("Email: Security breach detected in main database, immediate action required.", context_user))

Urgency: High


In [14]:
def test_classification(context, test_cases):
    results = []
    for test in test_cases:
        response = return_OAIResponse(test, context)
        results.append(f"Input: {test}\nResponse: {response}\n")
    return results

# Test cases para cada versión
movie_tests = [
    "Plot: A wizard learns magic at a special school while facing a dark enemy.",
    "Plot: A car chase through city streets ends in an explosive showdown.",
    "Plot: A chef works to earn a prestigious culinary award."
]

recipe_tests = [
    "Recipe: Boil water, add instant noodles, wait 3 minutes.",
    "Recipe: Create a 5-layer wedding cake with fondant decorations.",
    "Recipe: Make a basic tomato sauce with garlic and herbs."
]

email_tests = [
    "Email: Reminder about casual Friday dress code this week.",
    "Email: Critical system update needed in next 2 hours.",
    "Email: New employee starting next month."
]


### Casos de Prueba
- Películas con múltiples géneros
- Recetas de dificultad intermedia
- Correos con diferentes niveles de urgencia

## Lecciones Aprendidas

1. **Diseño de Prompts**:
   - La importancia de ejemplos claros y distintivos
   - El valor de mantener un formato consistente
   - La necesidad de cubrir diferentes casos

2. **Optimización del Modelo**:
   - El impacto de la temperatura en la consistencia
   - La importancia del contexto en la clasificación
   - El balance entre precisión y variabilidad

3. **Mejores Prácticas**:
   - Usar categorías mutuamente excluyentes cuando sea posible
   - Proporcionar ejemplos diversos pero claros
   - Mantener un formato consistente en los ejemplos

## Conclusiones

La implementación de diferentes clasificadores demostró que:

1. El Few-Shot Learning es efectivo para tareas de clasificación específicas
2. La calidad de los ejemplos impacta directamente en la precisión
3. Tres ejemplos bien elegidos son suficientes en la mayoría de los casos

## Recomendaciones

1. Implementar validación de resultados
2. Desarrollar conjuntos de prueba más extensos
3. Experimentar con diferentes números de ejemplos
4. Considerar la implementación de un sistema de retroalimentación
5. Documentar casos edge para futuras mejoras